**Author:** Javier Lendínez Castillo  
**License:** GNU Affero General Public License v3.0 ([LICENSE](../../LICENSE))

# Steady VLM Fundamentals

In this notebook, I reproduce and extend the high-aspect-ratio rectangular flat-plate case from Section 6.1 of my bachelor's thesis. The goal is to check that the implementation captures the expected linear trend for a symmetric, thin lifting surface and to examine how the solution changes as the discretization is refined.

**Assumptions and scope**:
- Steady, incompressible, inviscid, irrotational flow at low Mach number.
- Thin lifting surfaces, attached flow, and moderate angles.
- The solver does not calculate viscous drag or predict stall, separation, or compressibility effects.

**Conventions used:**
- `alpha`, `beta`, `sweep`, and `dihedral` are specified in radians when constructing the object and geometry.
- `span_fraction` is the total span of a symmetric section; the geometry uses half of it on each side.
- `n` and `m` are the spanwise and chordwise panel counts, respectively.

Thin-airfoil theory gives $dC_L/d\alpha = 2\pi$ per radian in the two-dimensional limit. Comparing against this value is a plausibility check for a wing with $AR=100$, not experimental validation or a claim that a finite wing must exactly reach the 2D limit.

## Theory and Workflow

The vortex-lattice method represents a thin lifting surface by panels carrying bound vortices and a prescribed trailing wake. At each control point, the normal component of the freestream and induced velocities must satisfy flow tangency. Discretization gives the system

$$\sum_j P_{ij}\Gamma_j=-\mathbf V_\infty\cdot\mathbf n_i,$$

where $P_{ij}$ is the normal velocity induced at panel $i$ by a unit-strength vortex $j$ and $\Gamma_j$ is its unknown circulation. The circulation then determines panel loads; the non-dimensional lift coefficient uses $q_\infty=\frac12\rho U^2$ and $C_L=L/(q_\infty S)$. The reported drag is induced drag from this inviscid model, not total aircraft drag.

For a symmetric thin airfoil at small incidence, $C_L\approx2\pi\alpha$ in the two-dimensional limit, with $\alpha$ in radians. A finite wing has tip effects, so this is a limiting comparison rather than an exact finite-wing solution. See [NASA Glenn: Lift Equation](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/lift-equation/) and [Leishman: Lifting-Line and Finite-Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/).

**How to use this notebook:** run the cells in order from a kernel whose working directory is inside the repository. First create the geometry and call `VLM.run_vlm()`, then use `compute_coefficients_vs_alpha()` for a sweep. The later cells inspect the spanwise distribution, mesh sensitivity, and induced-drag polar. The shape and finite-value checks verify implementation consistency; the thin-airfoil comparison is a theoretical plausibility check, not experimental validation.

In [30]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    current = start
    while current != current.parent:
        if (current / 'src' / 'vlm').is_dir():
            return current
        current = current.parent
    raise RuntimeError('Could not locate the VLMPy repository root (expected src/vlm).')


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_VLM = REPO_ROOT / 'src' / 'vlm'
if str(SRC_VLM) not in sys.path:
    sys.path.insert(0, str(SRC_VLM))

print(f'Repository root: {REPO_ROOT}')
print(f'VLM source path added to sys.path: {SRC_VLM}')

Repository root: c:\Users\javil\Documents\GitHub\VLMPy
VLM source path added to sys.path: c:\Users\javil\Documents\GitHub\VLMPy\src\vlm


## 1. Dependencies and imports

`lib.vlm` also imports `lib.geometry`, which requires `numpy-stl` when the module is loaded because of its STL export utilities. Although this analysis does not export meshes, that dependency must be available. If it is missing, install it with `pip install numpy plotly numpy-stl`.

In [31]:
import contextlib
import io

import numpy as np
import plotly.graph_objects as go

from lib.vlm import VLM, plot_wing_lift
from lib.vlm import VLM
from lib.geometry import (
    plot_wing_geometry_2d,
    plot_wing_discretization_2d,
    plot_wing_discretization_3d,
)

## 2. Study Case: High-Aspect-Ratio Flat Plate

I define a rectangular section with unit chord and a total span of 100 m. Therefore, the reference area is $S=100\,\mathrm{m^2}$ and the geometric aspect ratio is $AR=b^2/S=100$. I use the symmetric NACA 0106 airfoil as a thin-section approximation; the airfoil geometry is needed to calculate the camber slope even though the case is nominally flat.

I keep airspeed and air density constant throughout the comparisons. Thus, coefficient changes come from angle of attack or discretization, not from changes in flight conditions.

In [32]:
plane = {
    'wing_sections': [
        {
            'chord_root': 1.0,
            'chord_tip': 1.0,
            'span_fraction': 100.0,  # total span [m]; halved internally (symmetric wing)
            'sweep': 0.0,            # rad
            'dihedral': 0.0,         # rad
            'NACA_root': '0106',     # symmetric, zero-camber section (thin flat plate)
            'NACA_tip': '0106',
        }
    ]
}

U_INF = 50.0    # m/s
RHO = 1.225     # kg/m^3, ISA sea level
N_SPAN = 100    # spanwise panels
M_CHORD = 10    # chordwise panels

vlm = VLM(plane=plane, u=U_INF, rho=RHO, alpha=0.0, beta=0.0, n=N_SPAN, m=M_CHORD)

In [33]:
from lib.geometry import plot_wing_geometry
from lib.naca import plot_naca_airfoil

section = plane['wing_sections'][0]

# Plot the root airfoil
root_figure = go.Figure()
plot_naca_airfoil(
    root_figure,
    NACA=section['NACA_root'],
    cuerda=section['chord_root'],
    alpha=0.0,
)
root_figure.show()

# Plot the tip airfoil
tip_figure = go.Figure()
plot_naca_airfoil(
    tip_figure,
    NACA=section['NACA_tip'],
    cuerda=section['chord_tip'],
    alpha=0.0,
)
tip_figure.show()

# Calculate and plot the full wing geometry
vlm.calculate_geometry()
vlm.calculate_discretization()
fig_planform = go.Figure()
fig_planform = plot_wing_geometry_2d(fig_planform, vlm.wing_geometry, legend='Wing Geometry')
fig_planform.show()

Generating NACA 0106 airfoil with chord 1.0 m and angle of attack 0.00 rad


Generating NACA 0106 airfoil with chord 1.0 m and angle of attack 0.00 rad


## 3. Single-Point Solution and Numerical Checks

`run_vlm()` executes the geometry, discretization, and solution stages. I capture the diagnostic messages printed internally by the library so the notebook output focuses on the results.

The checks below verify dimensions and finite values for the influence matrix and circulation. They are consistency checks; by themselves, they do not prove that the model captures all of the wing's physics.

In [34]:
vlm.alpha = np.radians(5.0)

_captured = io.StringIO()
with contextlib.redirect_stdout(_captured):
    vlm.run_vlm()

print(f'CL = {vlm.CL:.4f}')
print(f'CD = {vlm.CD:.5f}')
print(f'Panels: {len(vlm.panel_data)} (n={vlm.n}, m={vlm.m})')

# Verification: shapes and finiteness (docs/README-performance.md procedure)
assert vlm.P_ij.shape == (len(vlm.panel_data), len(vlm.panel_data))
assert np.isfinite(vlm.P_ij).all()
assert np.isfinite(vlm.gammas).all()
assert np.isfinite(vlm.CL) and np.isfinite(vlm.CD)
print('All finite-value and shape checks passed.')

CL = 0.5305
CD = 0.00122
Panels: 1000 (n=100, m=10)
All finite-value and shape checks passed.


## 4. Lift Curve $C_L(\alpha)$ and Comparison with $2\pi$

`compute_coefficients_vs_alpha()` now returns the sweep dictionary and stores it in `vlm.results`. I calculate the coefficients from $-10^\circ$ to $10^\circ$, fit a straight line using the angle in radians, and compare its slope with the thin-airfoil reference. This range checks the approximately linear trend for this case; it should not be extrapolated to high angles or interpreted as a stall model.

In [35]:
angles_deg = [-10.0, -5.0, 0.0, 5.0, 10.0]

_captured = io.StringIO()
with contextlib.redirect_stdout(_captured):
    sweep_results = vlm.compute_coefficients_vs_alpha(angles_deg)

if sweep_results is None:
    sweep_results = vlm.results
else:
    assert sweep_results is vlm.results

angles_deg_arr = np.array(sweep_results['angles_deg'])
cl_arr = np.array(sweep_results['CL'])
angles_rad_arr = np.radians(angles_deg_arr)

slope_rad, intercept = np.polyfit(angles_rad_arr, cl_arr, 1)
theoretical_slope = 2.0 * np.pi
relative_error = abs(slope_rad - theoretical_slope) / theoretical_slope

print(f'Fitted lift-curve slope: {slope_rad:.4f} rad^-1')
print(f'Thin-airfoil limit (2*pi): {theoretical_slope:.4f} rad^-1')
print(f'Relative error: {relative_error * 100:.2f} %')

Fitted lift-curve slope: 6.0637 rad^-1
Thin-airfoil limit (2*pi): 6.2832 rad^-1
Relative error: 3.49 %


In [36]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=angles_deg_arr, y=cl_arr, mode='lines+markers', name='VLM'))
fig.add_trace(go.Scatter(
    x=angles_deg_arr,
    y=theoretical_slope * angles_rad_arr,
    mode='lines',
    name='2*pi*alpha (theory)',
    line=dict(dash='dash'),
))
fig.update_layout(
    title='Lift coefficient vs. angle of attack (AR = 100 flat plate)',
    xaxis_title='Angle of attack [deg]',
    yaxis_title='CL [-]',
    template='plotly_white',
)
fig.show()

### Diagnostic: Departure from the Local Linear Fit

The existing sweep can also be inspected through $\Delta C_L=C_L-(a\alpha+C_{L0})$. The residual describes the fit over the chosen angles, including changes in the solver response; it is not an experimental uncertainty estimate or a stall prediction.

In [37]:
fit_residuals = cl_arr - (slope_rad * angles_rad_arr + intercept)
assert np.isfinite(fit_residuals).all()
fig_fit_residuals = go.Figure(go.Scatter(
    x=angles_deg_arr, y=fit_residuals, mode='lines+markers', name='Fit residual'
))
fig_fit_residuals.add_hline(y=0, line_dash='dash')
fig_fit_residuals.update_layout(
    title='Lift-curve residual from the fitted line',
    xaxis_title='Angle of attack [deg]', yaxis_title='CL residual [-]',
    template='plotly_white',
)
fig_fit_residuals.show()

## 5. Spanwise $C_L$ Distribution at $\alpha=5^\circ$

I plot the local distribution along the span. I expect smaller variation near the center than near the tips, where three-dimensional finite-wing effects are more significant. The local distribution should not be confused with the wing's global coefficient.

In [38]:
vlm.alpha = np.radians(5.0)
_captured = io.StringIO()
with contextlib.redirect_stdout(_captured):
    vlm.run_vlm()

fig_span = go.Figure()
fig_span = plot_wing_lift(
    fig_span, vlm, vlm.CL_locals_wing.copy(),
    title='Wing CL Distribution', legend='CL [-]'
)
fig_span.show()

## 6. Limits of Interpretation

- The $2\pi$ limit belongs to linear two-dimensional thin-airfoil theory. The example wing has finite span, so it exhibits tip effects and discretization error.
- $AR=100$ approximates the two-dimensional case but does not represent a typical aircraft wing.
- This model does not represent viscosity, separation, stall, or compressibility. Therefore, the curves should not be extrapolated to those phenomena.

The next cell adds a mesh comparison at $\alpha=5^\circ$: geometry, airspeed, density, and `m` remain fixed while only `n` increases. The $2\pi\alpha$ reference is shown as the analytical 2D limit; the remaining difference includes finite-wing and discretization effects. This is a numerical sensitivity study, not independent validation.

### 7. Sensitivity of $C_L$ to Spanwise Refinement

To isolate spanwise discretization effects, I solve the case at $\alpha=5^\circ$ with `m=6` and six even values of `n`. I use even values because the discretization routine may automatically increase `n` and `m` when it cannot match the camber data to the panels. The cell reports and checks the effective values used by the solver. The $2\pi\alpha$ reference is a 2D limit, not the exact value for this finite wing.

In [39]:
import contextlib
import io
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

current = Path.cwd()
while current != current.parent and not (current / 'src' / 'vlm').is_dir():
    current = current.parent
if not (current / 'src' / 'vlm').is_dir():
    raise RuntimeError('Could not locate the VLMPy repository root.')
sys.path.insert(0, str(current / 'src' / 'vlm'))

plane_mesh = {
    'wing_sections': [{
        'chord_root': 1.0, 'chord_tip': 1.0,
        'span_fraction': 100.0, 'sweep': 0.0, 'dihedral': 0.0,
        'NACA_root': '0106', 'NACA_tip': '0106',
    }]
}
n_values = [10, 12, 14, 16, 18, 20]
m_fixed = 6
cl_values = []
effective_n_values = []
effective_m_values = []
alpha_mesh = np.radians(5.0)

for requested_n in n_values:
    case = VLM(plane_mesh, 50.0, 1.225, alpha_mesh, 0.0, requested_n, m_fixed)
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    effective_n_values.append(case.n)
    effective_m_values.append(case.m)
    cl_values.append(float(case.CL))
    print(
        'Requested n={}, m={}; effective n={}, m={}: CL={:.6f}'.format(
            requested_n, m_fixed, case.n, case.m, case.CL
        )
    )

thin_airfoil_cl = 2.0 * np.pi * alpha_mesh
mesh_figure = go.Figure()
mesh_figure.add_trace(go.Scatter(
    x=effective_n_values, y=cl_values, mode='lines+markers', name='VLMPy',
))
mesh_figure.add_trace(go.Scatter(
    x=effective_n_values, y=[thin_airfoil_cl] * len(effective_n_values),
    mode='lines', name='2*pi*alpha limit',
))
mesh_figure.update_layout(
    title='CL convergence with spanwise refinement',
    xaxis_title='Effective spanwise panel count (n)',
    yaxis_title='CL [-]', template='plotly_white',
)
mesh_figure.show()
assert effective_n_values == n_values
assert effective_m_values == [m_fixed] * len(n_values)
assert np.isfinite(cl_values).all()
assert np.isfinite(thin_airfoil_cl)

Requested n=10, m=6; effective n=10, m=6: CL=0.536046
Requested n=12, m=6; effective n=12, m=6: CL=0.535556
Requested n=14, m=6; effective n=14, m=6: CL=0.535142
Requested n=16, m=6; effective n=16, m=6: CL=0.534785
Requested n=18, m=6; effective n=18, m=6: CL=0.534472
Requested n=20, m=6; effective n=20, m=6: CL=0.534192


### Interpretation of Results

With the main mesh `n=100`, `m=10`, the linear fit to the $C_L(\alpha)$ curve gives $6.0637\,\mathrm{rad^{-1}}$, 3.49% below $2\pi$. This result is consistent with the expected trend for a finite-wing solution, where the slope is slightly reduced due to three-dimensional effects.

With `m=6`, $C_L$ decreases from 0.536046 (`n=10`) to 0.534192 (`n=20`), and the successive changes become smaller. The finest mesh is 2.58% below $2\pi\alpha=0.548311$. This trend indicates convergence toward the finite-wing solution, not the 2D limit: in this case, $2\pi\alpha$ is an upper reference because it omits tip effects. The mesh study and slope fit use different discretizations, so their values should not be compared directly.

### 8. Induced Polar $C_L$–$C_D$

To complete the lift curve, I compare lift and induced drag for several angles. In this calculation, $C_L$ changes sign with $\alpha$, while $C_D$ remains non-negative and is symmetric for opposite angles. At $\alpha=0^\circ$, both coefficients vanish for this symmetric geometry. `CD` is induced drag; it does not include skin-friction or form drag.

In [40]:
alpha_polar_deg = [-10, -5, 0, 5, 10]
cl_polar = []
cd_polar = []

for alpha_deg in alpha_polar_deg:
    case = VLM(plane_mesh, 50.0, 1.225, np.radians(alpha_deg), 0.0, 40, 6)
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    cl_polar.append(float(case.CL))
    cd_polar.append(float(case.CD))

polar_figure = go.Figure()
polar_figure.add_trace(go.Scatter(
    x=cd_polar, y=cl_polar, text=['{}°'.format(a) for a in alpha_polar_deg],
    mode='lines+markers+text', textposition='top center', name='VLMPy',
))
polar_figure.update_layout(
    title='Induced polar for a high-aspect-ratio flat plate',
    xaxis_title='Induced CD [-]',
    yaxis_title='CL [-]',
    template='plotly_white',
)
polar_figure.show()
assert np.isfinite(cl_polar).all() and np.isfinite(cd_polar).all()

## Finite-Wing Cross-Check and Polar Symmetry

The $2\pi$ slope is a two-dimensional limit. As an additional **theoretical comparison**, solve Prandtl's lifting-line equation for the same unswept rectangular planform and $a_0=2\pi$ rad$^{-1}$. 

The 40- and 80-mode results check the reference discretization; the VLM slope uses the existing $\pm5^\circ$ results on the main mesh. This compares different ideal-flow models and discretizations, not measured data or an experimental validation. The separate polar symmetry checks use the `n=40, m=6` cases already computed.

The Fourier formulation and its assumptions are described by [Leishman, Lifting-Line and Finite-Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/), equations 35 and 89–92. The high aspect ratio, ideal section slope, and finite-angle comparison limit its interpretation.

In [41]:
def rectangular_lifting_line_slope(span, chord, modes):
    theta = np.arange(1, modes + 1) * np.pi / (modes + 1)
    indices = np.arange(1, modes + 1)
    matrix = np.sin(theta[:, None] * indices[None, :]) * (
        4 * span / (2 * np.pi * chord) + indices[None, :] / np.sin(theta[:, None])
    )
    series = np.linalg.solve(matrix, np.ones(modes))
    return np.pi * span / chord * series[0]


slope_reference_40 = rectangular_lifting_line_slope(100.0, 1.0, 40)
slope_reference_80 = rectangular_lifting_line_slope(100.0, 1.0, 80)
positive_index = int(np.flatnonzero(angles_deg_arr == 5.0)[0])
negative_index = int(np.flatnonzero(angles_deg_arr == -5.0)[0])
slope_centered = (cl_arr[positive_index] - cl_arr[negative_index]) / np.deg2rad(10.0)
cl_symmetry_error = np.max(np.abs(np.asarray(cl_polar) + np.asarray(cl_polar)[::-1]))
cd_symmetry_error = np.max(np.abs(np.asarray(cd_polar) - np.asarray(cd_polar)[::-1]))
assert np.isfinite([slope_reference_40, slope_reference_80, slope_centered]).all()
assert cl_symmetry_error < 1e-10 and cd_symmetry_error < 1e-10
print(f'VLM centered slope (±5°): {slope_centered:.6f} rad⁻¹')
print(f'Rectangular lifting-line slope (80 modes): {slope_reference_80:.6f} rad⁻¹')
print(f'VLM/reference relative difference: {abs(slope_centered - slope_reference_80) / abs(slope_reference_80):.2%}')
print(f'Reference change from 40 to 80 modes: {abs(slope_reference_80 - slope_reference_40):.3e} rad⁻¹')
print(f'Polar symmetry: max |CL(α)+CL(-α)|={cl_symmetry_error:.3e}, max |CD(α)-CD(-α)|={cd_symmetry_error:.3e}')

VLM centered slope (±5°): 6.078769 rad⁻¹
Rectangular lifting-line slope (80 modes): 6.082305 rad⁻¹
VLM/reference relative difference: 0.06%
Reference change from 40 to 80 modes: 7.490e-05 rad⁻¹
Polar symmetry: max |CL(α)+CL(-α)|=0.000e+00, max |CD(α)-CD(-α)|=0.000e+00


## References

- [NASA Glenn: Lift Equation](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/lift-equation/).
- [Leishman: Lifting-Line and Finite-Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/).